# Coding Theory in SageMath: Reed–Solomon → Goppa → McEliece

This notebook is a computational study path for coding theory using SageMath.

The goal is not maximum performance. The goal is mathematical transparency:

1. Build finite fields and polynomial rings.
2. Construct Reed–Solomon codes by evaluation.
3. Decode Reed–Solomon codes by interpolation with errors.
4. Construct a classical binary Goppa code.
5. Decode a toy binary Goppa code using syndrome search.
6. Use the Goppa code inside a toy McEliece cryptosystem.

The final McEliece section is educational only. The parameters here are far too small for security.

## 0. SageMath setup

Run this notebook in SageMath, for example:

```bash
sage -n jupyter
```

The code below uses Sage objects such as `GF`, `PolynomialRing`, `matrix`, and vector spaces over finite fields.

In [ ]:
from itertools import combinations
import random

set_random_seed(20260430)

## 1. Finite fields and polynomial rings

Coding theory over finite fields starts with objects such as

\[
\mathbb F_q, \qquad \mathbb F_q[x], \qquad \mathbb F_q^n.
\]

Reed–Solomon codes use evaluation of polynomials. Goppa codes use rational functions and congruences modulo a Goppa polynomial.

In [ ]:
q = 11
F = GF(q)
R.<x> = PolynomialRing(F)

print("Field:", F)
print("Polynomial ring:", R)

f = 3 + 7*x + x^3
print("f(x) =", f)
print("f(2) =", f(F(2)))

## 2. Reed–Solomon codes by evaluation

Fix distinct evaluation points

\[
\alpha_1,\ldots,\alpha_n \in \mathbb F_q.
\]

The Reed–Solomon code of dimension \(k\) is

\[
RS_q(n,k)=\{(f(\alpha_1),\ldots,f(\alpha_n)) : f\in \mathbb F_q[x],\ \deg f<k\}.
\]

It has parameters

\[
[n,k,d]_q=[n,k,n-k+1]_q.
\]

So it is MDS: it meets the Singleton bound.

In [ ]:
def reed_solomon_generator_matrix(F, alpha, k):
    """Generator matrix for RS code using monomial basis 1,x,...,x^{k-1}."""
    return matrix(F, [[a^i for a in alpha] for i in range(k)])

n = 8
k = 4
alpha = list(F)[:n]
G_RS = reed_solomon_generator_matrix(F, alpha, k)

print("Evaluation points:", alpha)
print("Generator matrix G_RS:")
show(G_RS)
print("rank =", G_RS.rank())

In [ ]:
def rs_encode(message, G):
    return vector(G.base_ring(), message) * G

m = vector(F, [3, 1, 4, 1])
c = rs_encode(m, G_RS)

print("message =", m)
print("codeword =", c)

The message vector \((m_0,\ldots,m_{k-1})\) represents the polynomial

\[
f(x)=m_0+m_1x+\cdots +m_{k-1}x^{k-1}.
\]

Encoding means evaluating \(f\) at the chosen points.

In [ ]:
f = sum(m[i]*x^i for i in range(k))
print("message polynomial f(x) =", f)
print("evaluations =", vector(F, [f(a) for a in alpha]))

## 3. Reed–Solomon decoding by brute-force interpolation

For teaching purposes, we first implement a simple decoder.

If at most

\[
t=\left\lfloor \frac{n-k}{2}\right\rfloor
\]

errors occur, then one can recover the polynomial by trying subsets of size \(k\), interpolating, and checking how many positions agree.

This is not efficient for large \(n\), but it makes the idea completely visible.

In [ ]:
def interpolate_from_points(F, points):
    """Return polynomial f over F interpolating given points (a,b)."""
    R.<z> = PolynomialRing(F)
    return R.lagrange_polynomial(points)


def rs_decode_bruteforce(F, alpha, received, k):
    """Decode RS by trying all k-subsets. Educational, not efficient."""
    n = len(alpha)
    t = (n-k)//2
    R.<z> = PolynomialRing(F)
    for I in combinations(range(n), k):
        pts = [(F(alpha[i]), F(received[i])) for i in I]
        f = R.lagrange_polynomial(pts)
        if f == 0 or f.degree() < k:
            candidate = vector(F, [f(F(a)) for a in alpha])
            if sum(1 for i in range(n) if candidate[i] != received[i]) <= t:
                msg = vector(F, [f[i] for i in range(k)])
                return msg, candidate, f
    raise ValueError("Decoding failed")

# Add two errors, since floor((8-4)/2)=2.
r = vector(F, c)
r[1] += F(5)
r[6] += F(7)

msg_hat, c_hat, f_hat = rs_decode_bruteforce(F, alpha, r, k)
print("received =", r)
print("decoded polynomial =", f_hat)
print("decoded message =", msg_hat)
print("decoded codeword =", c_hat)
print("success =", c_hat == c)

## 4. Efficient Reed–Solomon decoding viewpoint

The brute-force decoder above is conceptually clean but inefficient.

The standard algebraic idea is to find an error-locator polynomial

\[
\Lambda(x)=\prod_{i\in E}(x-\alpha_i),
\]

where \(E\) is the set of error positions. Then

\[
\Lambda(\alpha_i)r_i=\Lambda(\alpha_i)f(\alpha_i)
\]

for every position \(i\):

- if \(i\notin E\), then \(r_i=f(\alpha_i)\);
- if \(i\in E\), then \(\Lambda(\alpha_i)=0\).

Thus the corrupted evaluation problem becomes a polynomial reconstruction problem.

## 5. Classical binary Goppa codes

Let

\[
L=(\alpha_1,\ldots,\alpha_n)
\]

be distinct elements of \(\mathbb F_{2^m}\), and let

\[
g(x)\in \mathbb F_{2^m}[x]
\]

be a polynomial such that \(g(\alpha_i)\neq 0\) for all \(i\).

The classical binary Goppa code is

\[
\Gamma(L,g)=\left\{c\in \mathbb F_2^n:
\sum_{i=1}^n \frac{c_i}{x-\alpha_i}\equiv 0 \pmod{g(x)}\right\}.
\]

Although the defining congruence lives over \(\mathbb F_{2^m}\), the code itself is binary.

In [ ]:
# Work over K = GF(2^m)
m_ext = 4
K.<a> = GF(2^m_ext)
S.<z> = PolynomialRing(K)

print("K =", K)
print("primitive element a =", a)

We choose a small squarefree Goppa polynomial \(g\) and a support \(L\) avoiding its roots.

In [ ]:
def random_squarefree_goppa_polynomial(K, t, max_tries=1000):
    S.<z> = PolynomialRing(K)
    for _ in range(max_tries):
        coeffs = [K.random_element() for _ in range(t)] + [K(1)]
        g = S(coeffs)
        if g.degree() == t and gcd(g, g.derivative()) == 1:
            return g
    raise RuntimeError("Could not find squarefree polynomial")

t = 3
g = random_squarefree_goppa_polynomial(K, t)
L = [u for u in K if g(u) != 0]

print("g(z) =", g)
print("degree t =", g.degree())
print("support length n =", len(L))
print("first support elements =", L[:8])

## 6. Parity-check matrix of a binary Goppa code

The congruence condition can be converted into a parity-check matrix.

A common parity-check matrix over \(\mathbb F_{2^m}\) is

\[
H_{j,i}=\frac{\alpha_i^j}{g(\alpha_i)},
\qquad
0\le j<t,
\quad
1\le i\le n.
\]

To obtain a binary parity-check matrix, expand each entry of \(\mathbb F_{2^m}\) in a basis over \(\mathbb F_2\).

In [ ]:
def goppa_parity_check_extension(K, L, g):
    """Parity-check matrix over K for a binary Goppa code."""
    t = g.degree()
    return matrix(K, [[(alpha^j) / g(alpha) for alpha in L] for j in range(t)])


def field_element_to_binary_vector(u, K):
    """Coordinate vector of u in the power basis over GF(2)."""
    return vector(GF(2), u.vector())


def expand_matrix_to_binary(HK, K):
    rows = []
    for row in HK.rows():
        expanded = [field_element_to_binary_vector(u, K) for u in row]
        m = K.degree()
        for ell in range(m):
            rows.append([expanded[j][ell] for j in range(HK.ncols())])
    return matrix(GF(2), rows)

HK = goppa_parity_check_extension(K, L, g)
Hbin = expand_matrix_to_binary(HK, K)

print("H over GF(2^m):", HK.nrows(), "x", HK.ncols())
print("H over GF(2):", Hbin.nrows(), "x", Hbin.ncols())
print("rank over GF(2) =", Hbin.rank())
show(Hbin)

The binary Goppa code is the kernel of this binary parity-check matrix:

\[
C=\ker(H_{\mathrm{bin}})\subseteq \mathbb F_2^n.
\]

In [ ]:
def code_from_parity_check(H):
    """Return a generator matrix whose rows span ker(H)."""
    return H.right_kernel().basis_matrix()

G_goppa = code_from_parity_check(Hbin)

print("Goppa code length n =", Hbin.ncols())
print("dimension k =", G_goppa.nrows())
print("redundancy rank(H) =", Hbin.rank())
print("designed correction capability t =", t)
show(G_goppa)

## 7. Syndrome decoding for a toy Goppa code

For real McEliece parameters, decoding uses a fast Goppa decoder such as Patterson decoding or a Reed–Solomon/alternant-style decoder.

For this educational notebook, we use syndrome search:

Given

\[
y=c+e,
\]

compute

\[
s=Hy^T=H e^T.
\]

If \(\operatorname{wt}(e)\le t\), we search all binary error vectors of weight at most \(t\) until the syndrome matches.

This is exponential in \(t\), but for tiny toy examples it is perfect for understanding the mechanism.

In [ ]:
def hamming_weight(v):
    return sum(1 for x in v if x != 0)


def binary_error_vector(n, positions):
    e = vector(GF(2), [0]*n)
    for i in positions:
        e[i] = 1
    return e


def syndrome_decode_binary(H, y, t):
    """Brute-force syndrome decoder for binary linear code. Educational only."""
    n = H.ncols()
    s = H * vector(GF(2), y)
    for w in range(t+1):
        for positions in combinations(range(n), w):
            e = binary_error_vector(n, positions)
            if H * e == s:
                return vector(GF(2), y) + e, e
    raise ValueError("No error vector of weight <= t found")

# Test Goppa encoding and decoding.
F2 = GF(2)
k_g = G_goppa.nrows()
n_g = G_goppa.ncols()

msg = vector(F2, [F2.random_element() for _ in range(k_g)])
c = msg * G_goppa
err_positions = random.sample(range(n_g), t)
e = binary_error_vector(n_g, err_positions)
y = c + e

c_dec, e_dec = syndrome_decode_binary(Hbin, y, t)

print("message =", msg)
print("error positions =", err_positions)
print("error vector =", e)
print("received y =", y)
print("decoded error =", e_dec)
print("decoded codeword correct =", c_dec == c)

## 8. Recovering the message from a generator matrix

After decoding the codeword \(c\), we need to recover the message \(m\) such that

\[
c=mG.
\]

If \(G\) is not systematic, we solve a linear system over \(\mathbb F_2\).

In [ ]:
def solve_message_from_codeword(G, c):
    """Solve mG=c for m over the base field."""
    F = G.base_ring()
    # mG=c is equivalent to G^T m^T = c^T.
    sol = G.transpose().solve_right(vector(F, c))
    return vector(F, sol)

msg_dec = solve_message_from_codeword(G_goppa, c_dec)
print("original message =", msg)
print("decoded message  =", msg_dec)
print("success =", msg_dec == msg)

## 9. Toy McEliece cryptosystem

The McEliece idea hides a structured code by scrambling its generator matrix.

Private key:

\[
G,\quad S,\quad P
\]

where

- \(G\) is a generator matrix of a decodable Goppa code;
- \(S\) is an invertible \(k\times k\) binary matrix;
- \(P\) is an \(n\times n\) permutation matrix.

Public key:

\[
G_{\mathrm{pub}}=SGP.
\]

Encryption:

\[
y=mG_{\mathrm{pub}}+e,
\qquad \operatorname{wt}(e)\le t.
\]

Decryption:

\[
yP^{-1}=mSG+eP^{-1}.
\]

Then decode using the private Goppa decoder, recover \(mS\), and multiply by \(S^{-1}\).

In [ ]:
def random_invertible_matrix(F, k):
    while True:
        M = random_matrix(F, k, k)
        if M.rank() == k:
            return M


def permutation_matrix_from_perm(F, perm):
    n = len(perm)
    P = zero_matrix(F, n, n)
    for i, j in enumerate(perm):
        P[i, j] = F(1)
    return P

S_scramble = random_invertible_matrix(F2, k_g)
perm = list(range(n_g))
random.shuffle(perm)
P_perm = permutation_matrix_from_perm(F2, perm)

G_pub = S_scramble * G_goppa * P_perm

print("Public generator matrix size:", G_pub.nrows(), "x", G_pub.ncols())
print("rank public matrix =", G_pub.rank())

In [ ]:
def mceliece_encrypt(m, G_pub, t):
    n = G_pub.ncols()
    positions = random.sample(range(n), t)
    e = binary_error_vector(n, positions)
    y = vector(GF(2), m) * G_pub + e
    return y, e, positions


def mceliece_decrypt(y, G, H, S, P, t):
    # Undo the permutation on the received word.
    P_inv = P.inverse()
    y_unperm = vector(GF(2), y) * P_inv
    
    # Decode in the secret Goppa code.
    c_unperm, e_unperm = syndrome_decode_binary(H, y_unperm, t)
    
    # Recover mS from c_unperm = (mS)G.
    m_times_S = solve_message_from_codeword(G, c_unperm)
    
    # Recover m.
    m = m_times_S * S.inverse()
    return m, c_unperm, e_unperm

m_plain = vector(F2, [F2.random_element() for _ in range(k_g)])
y_cipher, e_pub, pos_pub = mceliece_encrypt(m_plain, G_pub, t)

m_recovered, c_secret, e_secret = mceliece_decrypt(
    y_cipher, G_goppa, Hbin, S_scramble, P_perm, t
)

print("plaintext message =", m_plain)
print("public error positions =", pos_pub)
print("ciphertext =", y_cipher)
print("recovered message =", m_recovered)
print("success =", m_recovered == m_plain)

## 10. What this notebook proves computationally

You have now seen the complete computational chain:

\[
\text{polynomial evaluation}
\Rightarrow
\text{Reed–Solomon codes}
\Rightarrow
\text{Goppa parity checks}
\Rightarrow
\text{Goppa decoding}
\Rightarrow
\text{McEliece encryption/decryption}.
\]

Conceptually:

- Reed–Solomon codes are evaluation codes on \(\mathbb P^1\).
- Goppa codes are subfield subcodes of alternant codes.
- McEliece hides a code with an efficient private decoder.

The toy decoder here is brute force. For real parameters, replace syndrome search with a polynomial-time Goppa decoder.

## 11. Exercises

1. Change the Reed–Solomon parameters \((n,k)\). Verify that the decoder succeeds up to \(\lfloor(n-k)/2\rfloor\) errors.
2. Replace brute-force Reed–Solomon decoding with a Berlekamp–Welch decoder.
3. Try different Goppa polynomials \(g(z)\) of degree \(t\).
4. Increase \(m\) and \(t\). Observe when brute-force syndrome decoding becomes infeasible.
5. Convert the Goppa generator matrix into systematic form.
6. Implement Patterson decoding for binary Goppa codes.
7. Implement the alternant/Reed–Solomon-style decoder for binary Goppa codes.
8. Compare the public key sizes for different toy parameters.
9. Explain why this toy McEliece system is not secure.
10. Read Classic McEliece parameters and compare them with this notebook.